# Chapter 15 — Capstone: Governed Banking Complaint Agent

*Every prior chapter, wired into one system.*

## Objective

Assemble the complaint agent. Run it against the 20 synthetic eval cases. Walk through the audit log for three representative cases — a routine inquiry, an adversarial UDAAP complaint (an overdraft fee alleged to be unfair) and a PII-laden message — and produce a summary report.

In [ ]:
import json
from pathlib import Path

from glassloop.capstone import build_complaint_harness
from glassloop.core import Budget, BudgetTracker, TaskSpec
from proofloop.evaluation import summarize

root = Path('.')
if not (root / 'data' / 'policies').exists():
    root = Path('..')
policies_dir = root / 'data' / 'policies'
cases_path   = root / 'data' / 'eval_cases' / 'cases.json'
cases = json.loads(cases_path.read_text())
print(f'cases: {len(cases)}  policies: {sorted(p.name for p in policies_dir.glob("*.txt"))}')

## Build the harness

`build_complaint_harness` wires the agent, the five domain tools and the input policies (PII, prompt injection, prohibited advice, fee waiver) into a single `GovernanceHarness`.

All five tools are model-backed by a single open-weight Qwen2.5-3B-Instruct, each paired with a deterministic backstop so a regulated decision never rests on an unverifiable inference; each artifact loads on first use and caches for the process. `flag_regulatory`'s backstop is the richest: a separately trained, calibrated GMS store that verifies and corrects the model's proposed flags, *reads the message geometrically* through a fine-tuned embedding to catch the flags a keyword would miss, and decides escalation by walking the graph.

| Tool | Backed by | Artifact |
| --- | --- | --- |
| `classify_complaint` | Qwen encoder + LoRA adapter + classification head | `data/complaint_classifier_qwen/` |
| `extract_facts`      | Qwen + deterministic normalizer | `Qwen2.5-3B-Instruct` |
| `search_policy`      | Graph RAG retrieval + Qwen synthesis | `data/gms_policy_store/` |
| `flag_regulatory`    | Qwen proposes + GMS guard verifies/corrects + geometric reader | `data/gms_regulatory_store/`, `data/gms_regulatory_cap/` |
| `draft_response`     | Qwen + LoRA (complaints), template (inquiry / other) | `data/draft_response_lm_qwen/` |

In [ ]:
harness, registry = build_complaint_harness(policies_dir=policies_dir)
print('registered tools:', [t.name for t in registry.all()])

## Run a routine inquiry

In [ ]:
def did_escalate(traj):
    if traj.final_state.status in ('escalated', 'failed'): return True
    out = traj.final_state.final_output or {}
    return isinstance(out, dict) and out.get('recommended_action') == 'escalate'

case = next(c for c in cases if c['id'] == 'case-002')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
print('message:        ', case['message'])
print('status:         ', traj.final_state.status)
print('final output:   ', traj.final_state.final_output)

## Run an adversarial overdraft case — must escalate via UDAAP

In [ ]:
case = next(c for c in cases if c['id'] == 'case-016')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
print('message:    ', case['message'])
print('status:     ', traj.final_state.status)
esc = next(r for r in traj.records if r.action.kind == 'escalate')
print('escalation: ', esc.action.reason)

## Run a PII case — caught at the first tool call

In [ ]:
case = next(c for c in cases if c['id'] == 'case-011')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
failed_tool = next(r for r in traj.records if r.action.kind == 'tool_call' and not r.observation.get('success'))
print('message:           ', case['message'])
print('failing gate:      ', failed_tool.observation['error'])

## Fact extraction with a small open-weight LLM

`extract_facts` turns free text into a few structured fields — a natural fit for a language model, and the one place a brittle implementation quietly costs us downstream. Its `issue` field is what `flag_regulatory` keys on: UDAAP fires only when `issue` is `overdraft_fee`. A pure keyword extractor sets that from a substring ladder (`"overdraft"` → `overdraft_fee`, nothing else), so a fee complaint phrased without the word *overdraft* — *"reverse this \$35 charge on my checking account"* — falls through to `account_issue` and the flag never fires.

So `extract_facts` uses **Qwen2.5-3B-Instruct**, an off-the-shelf open-weight LLM, at greedy decoding (deterministic for a fixed input). It loads in tens of seconds and answers in ~1–2 s/message. Set `AGENTLAB_USE_LLM_EXTRACT=0` to force the pure-regex fallback (offline, no GPU).

In [ ]:
from glassloop.models.qwen_extractor import get_default_extractor

extract = get_default_extractor()
for msg in [
    "I'm going to sue you unless you give me my money back today.",   # case-009
    "I was charged twice for the same purchase, please reverse one.", # case-020
]:
    print(msg)
    print('   raw LLM ->', extract.extract(msg))

The raw model output can't drive a regulated decision directly. On case-009 — *"...give me my money back today"*, which names no product and no fee — the LLM **invents** `checking_account / overdraft_fee`, which would trip UDAAP and force a **false escalation** (escalation accuracy 100% → 95%).

The hybrid fixes this: the LLM *proposes*, a deterministic layer *disposes*. `_normalize_llm` coerces the model's `product` onto the enumerable taxonomy (falling back to the keyword result when off-taxonomy) and derives `issue` from it. Then two guards enforce a rule the model cannot override — **a regulatory flag may only fire on evidence present in the source text**:

```python
_FEE_SIGNAL  = re.compile(r"overdraft|\bfee\b|charge|\$\s*\d|revers", re.I)
_REGX_SIGNAL = re.compile(r"mortgage|escrow|home loan|\bloan\b", re.I)

# UDAAP guard: overdraft_fee only stands if the message shows a fee/charge.
if issue == "overdraft_fee" and not _FEE_SIGNAL.search(message):
    issue = "account_issue" if product == "checking_account" else "general"
# Reg-X guard: a mortgage/loan product only stands with textual evidence.
if product in ("mortgage", "loan") and not _REGX_SIGNAL.search(message):
    product = rule["product"]
```

On case-009 the UDAAP guard fires — *"money back today"* carries no fee token — `issue` drops to `account_issue`, UDAAP stays silent, and escalation returns to 100%. The model keeps its recall on the cases that *do* carry a fee signal; it just can't manufacture a regulatory event out of thin air. A language model is safe inside a governed workflow exactly when a deterministic gate decides what its output is allowed to *cause*.

## Aggregate over all cases

Escalation accuracy is the headline. Classification accuracy is a secondary metric — when an early gate denies a tool call, the classifier never runs. That is correct governed behavior; you cannot classify a message you refuse to process.

In [ ]:
classify_correct = 0
classifier_ran   = 0
escalation_correct = 0
for case in cases:
    task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
    traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
    out = traj.final_state.final_output or {}
    actual_class = out.get('classification') if isinstance(out, dict) else None
    if actual_class is None:
        for rec in traj.records:
            if rec.action.kind == 'tool_call' and rec.observation.get('success'):
                tool_out = rec.observation.get('output') or {}
                if isinstance(tool_out, dict) and 'category' in tool_out:
                    actual_class = tool_out['category']
                    break
    if actual_class is not None:
        classifier_ran += 1
    if actual_class == case.get('expected_classification'):
        classify_correct += 1
    if did_escalate(traj) == case.get('expected_escalation'):
        escalation_correct += 1

n = len(cases)
print(f'classification accuracy: {classify_correct}/{n} ({100*classify_correct/n:.0f}%)')
print(f'  of cases where the classifier ran: '
      f'{classify_correct}/{classifier_ran} ({100*classify_correct/classifier_ran:.0f}%)')
print(f'escalation accuracy:     {escalation_correct}/{n} ({100*escalation_correct/n:.0f}%)')
print(f'audit chain verifies:    {harness.audit.verify()}')

## Inspect the drafts

Complaint cases that classify as complaint and aren't escalated by `flag_regulatory` reach `draft_response`. The LoRA's output is short, on-template, and names the right policy with the right number.

In [ ]:
for case_id in ['case-003', 'case-005', 'case-020']:
    case = next(c for c in cases if c['id'] == case_id)
    task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
    traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
    out = traj.final_state.final_output or {}
    print(f'[{case_id}] {case["message"][:55]!r}')
    print(f'  -> {out.get("draft_response", "")!r}')
    print()

Graph RAG's alias lexicon routes customer vocabulary (`chargeback`, `unauthorized transaction`, `fee waiver`) onto canonical policy ids. The LoRA was SFT-trained on the same canonical-summary template the kg-memory store now serves, so retrieval and generation are aligned on the same six policies. The split between LoRA-for-complaints and template-for-inquiry is intentional: the LoRA's training distribution is complaint→reply only.

## What the agent does and does not do

- It **classifies, extracts, retrieves policy, drafts**, and **escalates** when warranted.
- It does **not** invent facts, promise fee waivers, or process messages containing unredacted PII.
- Every step is recorded in a tamper-evident audit chain.

Where this would fall short in production:
- The policy library is small — six short text files plus one structured compendium. Real systems compose dozens, with versioning.
- The alias lexicon is hand-curated; customer vocabulary outside it (raw `SSN`, slang, regional spellings) still misses.
- The draft generator is a 0.9M-parameter TinyGPT — grounded but short and structurally narrow.
- The synthetic corpus is small. Real corpora are thousands; chunking and ranking start to matter.

In [ ]:
# Self-check
assert escalation_correct == n, f'expected 100% escalation accuracy, got {escalation_correct}/{n}'
assert harness.audit.verify()
print('OK')